# W04 — Baseline Score: Ranking Signal Analysis

**Lane (confirmed):** Ranking Signal Analysis
**Month used:** `month=2026-03` (mid-panel, not the sealed final month)

> **Before you submit:** run every cell top-to-bottom in Colab with `HF_TOKEN` set as a Secret.
> All query and rule logic below is complete and ready to execute; the actual bucket counts,
> verdicts, and top-10 rows can only be produced by running it against the real gated warehouse
> data. Run All, read the printed verdicts and top-10 output, fill in the two short judgment lines
> flagged `# FILL AFTER RUN`, then commit.


## Setup

In [ ]:
import duckdb, os, json
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET hf_token='{os.environ['HF_TOKEN']}';")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

import pathlib
pathlib.Path("work/outputs").mkdir(parents=True, exist_ok=True)


## 1) Signal Checks — Two Signals Behind Real FlyRank Flags

Two signals feed the baseline rule below, both checked with a bucket table (with `n`) before
they're trusted:

- **Staleness** — behind the `stale_visible_page` refresh flag (`days_since_last_update >= 180`
  logic from the session).
- **CTR-vs-position** — behind the `low_ctr_visible_page` / CTR-fix logic (CTR should fall as
  position tier worsens; pages that undercut their own tier's typical CTR are the candidates).

Both are flag-linked, so either alone would satisfy the "at least one flag-linked" requirement —
using both gives the rule two independently-checked legs instead of one.


### Signal 1 — Staleness vs. decline (bucket table, n printed)

In [ ]:
staleness_check = con.sql(f"""
WITH latest_trend AS (
  SELECT content_hash_id, client_hash_id, trend_direction
  FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_hash_id, client_hash_id ORDER BY report_date DESC
  ) = 1
),
staleness AS (
  SELECT
    content_hash_id,
    CASE
      WHEN days_since_last_update < 90  THEN '0-89d'
      WHEN days_since_last_update < 180 THEN '90-179d'
      WHEN days_since_last_update < 365 THEN '180-364d'
      ELSE '365d+'
    END AS staleness_bucket
  FROM read_parquet('{BASE}/dim_content/*.parquet')
)
SELECT
  s.staleness_bucket,
  COUNT(*) AS n,
  AVG(CASE WHEN t.trend_direction = 'down' THEN 1.0 ELSE 0.0 END) AS decline_rate
FROM staleness s
JOIN latest_trend t USING (content_hash_id)
GROUP BY 1
ORDER BY 1
""").df()
staleness_check


**Verdict — staleness:** `# FILL AFTER RUN` — one word: CONFIRMED / OPPOSITE / MIXED / FALSE.
Read it straight off the table: CONFIRMED if `decline_rate` rises monotonically (or close to it)
from `0-89d` to `365d+`; OPPOSITE if it falls; MIXED if it's non-monotonic; FALSE if buckets show
no meaningful difference. A clean negative here is still a win — it means staleness alone
shouldn't drive the rule's weight.


### Signal 2 — CTR vs. position tier (bucket table, n printed)

In [ ]:
ctr_position_check = con.sql(f"""
WITH monthly AS (
  SELECT
    content_hash_id,
    client_hash_id,
    SUM(impressions) AS impressions_30d,
    SUM(clicks)       AS clicks_30d,
    AVG(position)     AS avg_position_30d
  FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
  GROUP BY 1, 2
  HAVING SUM(impressions) >= 100  -- minimum volume so tiny-sample noise doesn't skew the tier
)
SELECT
  CASE
    WHEN avg_position_30d <= 3  THEN '1-3'
    WHEN avg_position_30d <= 10 THEN '4-10'
    WHEN avg_position_30d <= 20 THEN '11-20'
    ELSE '21+'
  END AS position_tier,
  COUNT(*) AS n,
  AVG(clicks_30d * 1.0 / NULLIF(impressions_30d, 0)) AS avg_ctr
FROM monthly
GROUP BY 1
ORDER BY 1
""").df()
ctr_position_check


**Verdict — CTR-vs-position:** `# FILL AFTER RUN` — CONFIRMED if `avg_ctr` clearly declines
as position tier worsens (1-3 highest, 21+ lowest); OPPOSITE if it rises; MIXED if it's uneven;
FALSE if tiers look flat. This sets the tier baseline the rule below compares each page against.

**Rule reasoning:** the baseline score only uses a signal at full weight if its verdict is
CONFIRMED. If either check comes back OPPOSITE or FALSE, that signal's weight drops to 0 in the
rule below rather than being kept on faith — the whole point of checking first.


## 2) The Rule — Score, Reason Code, Action Label

**Candidate pool:** pages with `impressions_30d >= 250` (enough demand to matter — below this,
CTR and staleness signal is mostly noise).

**Score:**
```
baseline_action_score = w_stale * staleness_norm + w_ctr * ctr_gap_norm
```
- `staleness_norm` = `days_since_last_update / 365`, clipped to `[0, 1]`.
- `ctr_gap_norm` = `(tier_median_ctr - page_ctr) / tier_median_ctr`, clipped to `[0, 1]` (0 if the
  page already beats its tier).
- `w_stale`, `w_ctr` = 0.5 / 0.5 by default — set to 0 for either signal above that isn't
  CONFIRMED, then renormalize the remaining weight to 1.0.

**Reason code (one, applied to every row in the ranked queue):** `stale_underperforming_ctr` —
the page is both older than the flag-linked threshold and below its position tier's expected CTR.

**Action label:** `review_for_refresh` — the queue is a review-first list, not a guarantee a
refresh will work (see limitation in section 4).


In [ ]:
W_STALE = 0.5  # set to 0 above if staleness verdict != CONFIRMED, then renormalize
W_CTR   = 0.5   # set to 0 above if CTR-vs-position verdict != CONFIRMED, then renormalize
MIN_IMPRESSIONS = 250

queue = con.sql(f"""
WITH monthly AS (
  SELECT
    f.content_hash_id,
    f.client_hash_id,
    SUM(f.impressions) AS impressions_30d,
    SUM(f.clicks)       AS clicks_30d,
    AVG(f.position)     AS avg_position_30d,
    d.days_since_last_update AS days_since_last_update
  FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
  JOIN read_parquet('{BASE}/dim_content/*.parquet') d
    ON f.content_hash_id = d.content_hash_id
  GROUP BY 1, 2, d.days_since_last_update
  HAVING SUM(f.impressions) >= {MIN_IMPRESSIONS}
),
tiered AS (
  SELECT *,
    clicks_30d * 1.0 / NULLIF(impressions_30d, 0) AS ctr_30d,
    CASE
      WHEN avg_position_30d <= 3  THEN '1-3'
      WHEN avg_position_30d <= 10 THEN '4-10'
      WHEN avg_position_30d <= 20 THEN '11-20'
      ELSE '21+'
    END AS position_tier
  FROM monthly
),
tier_medians AS (
  SELECT position_tier, MEDIAN(ctr_30d) AS tier_median_ctr
  FROM tiered
  GROUP BY 1
)
SELECT
  t.content_hash_id,
  t.client_hash_id,
  t.impressions_30d,
  t.clicks_30d,
  ROUND(t.ctr_30d, 4)          AS ctr_30d,
  t.position_tier,
  t.days_since_last_update,
  LEAST(GREATEST(t.days_since_last_update / 365.0, 0), 1)                                    AS staleness_norm,
  LEAST(GREATEST((m.tier_median_ctr - t.ctr_30d) / NULLIF(m.tier_median_ctr, 0), 0), 1)       AS ctr_gap_norm
FROM tiered t
JOIN tier_medians m USING (position_tier)
""").df()

queue["baseline_action_score"] = (
    W_STALE * queue["staleness_norm"] + W_CTR * queue["ctr_gap_norm"]
)
queue["reason_code"] = "stale_underperforming_ctr"
queue["action"] = "review_for_refresh"

queue = queue.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue.head(10)


### Run receipts (commit this — the CSV itself stays out of git by design)

In [ ]:
receipts = {
    "lane": "ranking_signal_analysis",
    "month": MONTH,
    "min_impressions": MIN_IMPRESSIONS,
    "weights": {"staleness": W_STALE, "ctr_gap": W_CTR},
    "reason_code": "stale_underperforming_ctr",
    "action": "review_for_refresh",
    "n_candidate_rows": int(len(queue)),
    "n_clients": int(queue["client_hash_id"].nunique()),
    "score_summary": {
        "min": float(queue["baseline_action_score"].min()),
        "median": float(queue["baseline_action_score"].median()),
        "max": float(queue["baseline_action_score"].max()),
    },
    "staleness_verdict": "FILL_AFTER_RUN",
    "ctr_position_verdict": "FILL_AFTER_RUN",
}

with open("work/outputs/w04_baseline_score_metrics.json", "w") as f:
    json.dump(receipts, f, indent=2)

receipts


## 3) Top-10 Review

For each of the top 10 rows: the action, why it's there, and what would make it wrong. The first
two columns come straight from the rule; `what_would_make_it_wrong` is a generic per-row check —
read the actual top 10 after running and tighten these before committing.


In [ ]:
top10 = queue.head(10).copy()
top10["why_its_there"] = (
    "stale ("
    + (top10["days_since_last_update"].astype(str))
    + "d since update) + CTR "
    + top10["ctr_30d"].round(4).astype(str)
    + " vs tier " + top10["position_tier"] + " expectation, on "
    + top10["impressions_30d"].astype(str) + " impressions"
)
top10["what_would_make_it_wrong"] = (
    "wrong if a sibling page absorbed this page's demand (consolidation), the drop is seasonal "
    "for this topic, or the page was already updated after " + MONTH + " and this reflects stale metadata"
)

top10_review = top10[[
    "rank", "content_hash_id", "client_hash_id", "action",
    "baseline_action_score", "why_its_there", "what_would_make_it_wrong"
]]
top10_review


`# FILL AFTER RUN` — after reading the actual top 10, replace the generic
`what_would_make_it_wrong` line for any row where a *specific* look-alike (consolidation,
seasonality, recent-but-unlogged update) is genuinely plausible for that page, per section 7 of
the lane guide.


## 4) Weak Picks / Limitations

- **Threshold arbitrariness:** `MIN_IMPRESSIONS = 250` and the staleness buckets are chosen
  thresholds, not universal truths — a different capacity or risk tolerance would move them
  (section 11 of the lane guide).
- **No consolidation/seasonality check yet:** the rule doesn't look at sibling-page or
  site-level trends, so some top-ranked pages may be consolidation or seasonal cases rather than
  real decline candidates (see section 7). That check is a natural extension for
  `w04_signal_audit.ipynb`, not required here.
- **Not causal:** a high `baseline_action_score` means "review first," never "refreshing this
  page will recover traffic" — no experiment was run.
- **Single reason code:** every flagged row gets the same `stale_underperforming_ctr` tag even
  though the two underlying signals (staleness, CTR gap) can contribute unevenly per row; a future
  version could split this into per-signal reason codes.
- **This is the baseline Week 5 must beat** — it is intentionally simple and rule-based, not a
  ceiling.


## 5) Self-Check

- [ ] Two signal checks, each with a bucket table and printed `n` — section 1.
- [ ] At least one signal directly behind a real FlyRank flag (staleness → refresh flags,
      CTR-vs-position → CTR-fix logic) — both are, here.
- [ ] One-word verdict per signal (CONFIRMED / OPPOSITE / MIXED / FALSE) — filled after running.
- [ ] One rule encoded: score + one reason code + one action label — section 2.
- [ ] Ranked queue written to `work/outputs/baseline_action_score.csv` from the notebook itself.
- [ ] Run receipts committed to `work/outputs/w04_baseline_score_metrics.json`.
- [ ] Top-10 reviewed, one line each: action, why, what would make it wrong — section 3.
- [ ] No future-window or label-derived inputs used (all features are within-`month=2026-03`,
      and `trend_pct`/`trend_direction` are used only for the section-1 signal check, never as a
      rule input).
- [ ] Lane confirmed: Ranking Signal Analysis.
